# PDE Discovery: Kuramoto–Sivashinsky Equation

## Standard PDE-find vs. Equivariant PDE-find

We attempt to identify the **2D Kuramoto–Sivashinsky equation**
$$
f_t = -\tfrac{1}{2}|\nabla f|^2 - \Delta f - \Delta^2 f
$$
from simulation data.  
In the equivariant basis this is $-\tfrac{1}{2}N_7 - N_4 - N_{10}$.  
Note that $N_7 = |\nabla f|^2$ involves a *product of derivatives* and is
**absent from the standard coordinate library** — this is the key test case.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')
from multigraph_operators import (
    build_equivariant_library, build_standard_library,
    TrainSTRidge, print_pde, rotation_90,
    spectral_laplacian, spectral_grad
)

np.random.seed(42)
print('imports ok')


imports ok


## 1. Simulate 2D KS via spectral IMEX

The linear part is $L_k = K^2 - K^4$ (anti-diffusion at $K<1$, hyperdiffusion at $K>1$):
$$
\hat{f}^{n+1} = \frac{\hat{f}^n - \frac{\Delta t}{2}\,\widehat{|\nabla f|^2}^n}{1 - \Delta t\, L_k}
$$


In [2]:
N  = 64
L  = 16 * np.pi   # large domain: ~8 pattern wavelengths per side
x  = np.linspace(0, L, N, endpoint=False)
X, Y = np.meshgrid(x, x, indexing='ij')

k   = np.fft.fftfreq(N) * N * (2 * np.pi / L)
KX, KY = np.meshgrid(k, k, indexing='ij')
K2  = KX**2 + KY**2
K4  = K2**2

dt         = 0.05
T_end      = 400.0
n_steps    = int(T_end / dt)
save_every = 5        # dt_snap = 0.25

# Linear operator: -Δf - Δ²f -> K² - K⁴ in Fourier
Lk    = K2 - K4
denom = 1.0 - dt * Lk

# Small-amplitude random initial condition
f0 = 0.1 * np.random.randn(N, N)
print(f'Grid: {N}x{N}, L={L:.2f}, dt={dt}, T_end={T_end}, save_every={save_every}')
print(f'dt_snap = {dt * save_every}')


Grid: 64x64, L=50.27, dt=0.05, T_end=400.0, save_every=5
dt_snap = 0.25


In [3]:
snapshots = []
times     = []

f = f0.copy()
for step in range(n_steps):
    fx, fy   = spectral_grad(f, L, L)
    NL       = -0.5 * (fx**2 + fy**2)          # nonlinear: -½|∇f|²
    fhat     = np.fft.fft2(f)
    NL_hat   = np.fft.fft2(NL)
    fhat     = (fhat + dt * NL_hat) / denom
    f        = np.real(np.fft.ifft2(fhat))
    if step % save_every == 0:
        snapshots.append(f.copy())
        times.append(step * dt)

snapshots = np.array(snapshots)
times     = np.array(times)
print(f'Snapshots: {len(snapshots)},  f range: [{snapshots.min():.3f}, {snapshots.max():.3f}]')


Snapshots: 1600,  f range: [-1003.451, 2.210]


## 2. Visualise dynamics

In [4]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
plot_idx = [len(snapshots)//5, len(snapshots)//2, 3*len(snapshots)//4, -1]
vmax = max(abs(snapshots[i]).max() for i in plot_idx)
for ax, idx in zip(axes, plot_idx):
    im = ax.imshow(snapshots[idx], cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   origin='lower', extent=[0, L, 0, L])
    ax.set_title(f't = {times[idx]:.0f}', fontsize=13)
    ax.axis('off')
fig.colorbar(im, ax=axes[-1], shrink=0.8)
fig.suptitle('Kuramoto–Sivashinsky chaotic pattern on $[0, 16\pi]^2$', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/kuramoto_sivashinsky_dynamics.pdf', bbox_inches='tight')
plt.close()
print('saved dynamics')


saved dynamics


## 3. Extract data for regression

Skip the first 40% of snapshots (transient growth phase).


In [5]:
skip = max(1, int(0.4 * len(snapshots)))
snapshots_use = snapshots[skip:]
times_use     = times[skip:]

dt_snap  = times_use[1] - times_use[0]
ft_snaps = (snapshots_use[2:] - snapshots_use[:-2]) / (2 * dt_snap)
f_snaps  = snapshots_use[1:-1]
print(f'Using {len(f_snaps)} snapshots for regression (skipped first {skip})')
print(f'dt_snap = {dt_snap}')


Using 958 snapshots for regression (skipped first 640)
dt_snap = 0.25


## 4. Build the libraries

In [6]:
n_per_snap = 200
Theta_eq_rows, Theta_std_rows, Ft_all_rows = [], [], []

np.random.seed(0)
for s in range(len(f_snaps)):
    idx = np.random.choice(N * N, n_per_snap, replace=False)
    Ft_all_rows.append(ft_snaps[s].ravel()[idx])
    Th_eq,  desc_eq  = build_equivariant_library(f_snaps[s], L, L)
    Th_std, desc_std = build_standard_library(f_snaps[s], L, L, max_order=4, max_poly=3)
    Theta_eq_rows.append(Th_eq[idx])
    Theta_std_rows.append(Th_std[idx])

Ft_all    = np.concatenate(Ft_all_rows)
Theta_eq  = np.vstack(Theta_eq_rows).astype(float)
Theta_std = np.vstack(Theta_std_rows).astype(float)

print(f'Equivariant Theta: {Theta_eq.shape},  cond = {np.linalg.cond(Theta_eq):.2e}')
print(f'Standard   Theta: {Theta_std.shape},  cond = {np.linalg.cond(Theta_std):.2e}')


Equivariant Theta: (191600, 20),  cond = 3.07e+10
Standard   Theta: (191600, 60),  cond = 8.11e+13


## 5. Sparse regression (STRidge)

True PDE: $f_t = -\tfrac{1}{2}|\nabla f|^2 - \Delta f - \Delta^2 f$

Equivariant library indices: 4 ($\Delta f$) → $-1.0$, 7 ($|\nabla f|^2$) → $-0.5$, 10 ($\Delta^2 f$) → $-1.0$


In [7]:
lam   = 1e-5
d_tol = 0.05

print('=== Equivariant PDE-find ===')
w_eq = TrainSTRidge(Theta_eq, Ft_all, lam, d_tol)
print_pde(w_eq, desc_eq)

print('\n=== Standard PDE-find ===')
w_std = TrainSTRidge(Theta_std, Ft_all, lam, d_tol)
print_pde(w_std, desc_std)


=== Equivariant PDE-find ===
f_t = (-0.01142)1 + (+0.00007)f + (+0.00000)f^2 + (-0.99778)Δf + (-0.47943)|∇f|² + (-1.00809)Δ²f + (+0.02513)⟨∇Δf,∇f⟩ + (+0.00219)|∇²f|² + (-0.01374)∇²f(∇f,∇f)

=== Standard PDE-find ===
f_t = (+0.01863)1 + (-0.45422)f_y + (-0.35498)f_x + (+1.00747)f_yy + (+0.34040)f_xy + (-0.63759)f_xx + (+0.12172)f_yyy + (-0.78196)f_xyy + (+0.34919)f_xxy + (-0.13049)f_xxx + (-0.10029)f_yyyy + (+0.18787)f_xyyy + (+1.20182)f_xxyy + (-0.28767)f_xxxy + (-0.69922)f_xxxx + (+0.00005)f + (-0.00187)f·f_y + (-0.00255)f·f_x + (+0.00480)f·f_yy + (+0.00139)f·f_xy + (-0.00315)f·f_xx + (+0.00045)f·f_yyy + (-0.00383)f·f_xyy + (+0.00188)f·f_xxy + (-0.00099)f·f_xxx + (-0.00039)f·f_yyyy + (+0.00064)f·f_xyyy + (+0.00591)f·f_xxyy + (-0.00117)f·f_xxxy + (-0.00340)f·f_xxxx + (+0.00000)f^2 + (-0.00000)f^2·f_y + (-0.00001)f^2·f_x + (+0.00001)f^2·f_yy + (+0.00000)f^2·f_xy + (-0.00000)f^2·f_xx + (+0.00000)f^2·f_yyy + (-0.00001)f^2·f_xyy + (+0.00000)f^2·f_xxy + (-0.00000)f^2·f_xxx + (-0.00000)f^2·f

In [8]:
# True coefficient vector: N4=-1, N7=-0.5, N10=-1
true_eq = np.zeros(len(desc_eq))
true_eq[4]  = -1.0    # Δf
true_eq[7]  = -0.5    # |∇f|²
true_eq[10] = -1.0    # Δ²f

active_eq      = np.abs(true_eq) > 1e-10
coeff_error_eq = float(np.max(np.abs(w_eq[active_eq] - true_eq[active_eq])))
false_pos_eq   = int(np.sum((np.abs(w_eq)  > 1e-10) & ~active_eq))
false_neg_eq   = int(np.sum((np.abs(w_eq)  < 1e-10) &  active_eq))

print('--- Recovery summary ---')
print(f'True coefficients: Δf={true_eq[4]:.2f}, |∇f|²={true_eq[7]:.2f}, Δ²f={true_eq[10]:.2f}')
print(f'Equivariant: coeff_error = {coeff_error_eq:.3e}, '
      f'false_pos = {false_pos_eq}, false_neg = {false_neg_eq}')
print(f'Standard:    {int(np.sum(np.abs(w_std) > 1e-10))} active terms out of {len(w_std)}')


--- Recovery summary ---
True coefficients: Δf=-1.00, |∇f|²=-0.50, Δ²f=-1.00
Equivariant: coeff_error = 2.057e-02, false_pos = 6, false_neg = 0
Standard:    58 active terms out of 60


## 6. Equivariance test (90° rotation)

In [9]:
ft_snaps_r = (rotation_90(snapshots_use[2:].transpose(1,2,0)).transpose(2,0,1)
              - rotation_90(snapshots_use[:-2].transpose(1,2,0)).transpose(2,0,1)) / (2 * dt_snap)
f_snaps_r  = rotation_90(snapshots_use[1:-1].transpose(1,2,0)).transpose(2,0,1)

np.random.seed(0)
Ft_r, Theta_eq_r_rows, Theta_std_r_rows = [], [], []
for s in range(len(f_snaps_r)):
    idx = np.random.choice(N * N, n_per_snap, replace=False)
    Ft_r.append(ft_snaps_r[s].ravel()[idx])
    Th_eq_r,  _ = build_equivariant_library(f_snaps_r[s], L, L)
    Th_std_r, _ = build_standard_library(f_snaps_r[s], L, L, max_order=4, max_poly=3)
    Theta_eq_r_rows.append(Th_eq_r[idx])
    Theta_std_r_rows.append(Th_std_r[idx])

Ft_r        = np.concatenate(Ft_r)
Theta_eq_r  = np.vstack(Theta_eq_r_rows).astype(float)
Theta_std_r = np.vstack(Theta_std_r_rows).astype(float)

print('=== Equivariant on rotated data ===')
w_eq_r = TrainSTRidge(Theta_eq_r, Ft_r, lam, d_tol)
print_pde(w_eq_r, desc_eq)

print('\n=== Standard on rotated data ===')
w_std_r = TrainSTRidge(Theta_std_r, Ft_r, lam, d_tol)
print_pde(w_std_r, desc_std)

rot_change_eq  = float(np.max(np.abs(w_eq  - w_eq_r)))
rot_change_std = float(np.max(np.abs(w_std - w_std_r)))
print(f'\nCoefficient change under 90° rotation:')
print(f'  Equivariant  max|Δc| = {rot_change_eq:.2e}')
print(f'  Standard     max|Δc| = {rot_change_std:.2e}')


=== Equivariant on rotated data ===
f_t = (-0.03277)1 + (+0.00002)f + (+0.00000)f^2 + (-0.98068)Δf + (+0.00005)f·Δf + (+0.00000)f²·Δf + (-0.47340)|∇f|² + (+0.00002)f·|∇f|² + (+0.00000)f²·|∇f|² + (-0.99535)Δ²f + (+0.00003)f·Δ²f + (+0.00000)f²·Δ²f + (+0.02510)⟨∇Δf,∇f⟩ + (-0.00000)f·⟨∇Δf,∇f⟩ + (+0.00216)|∇²f|² + (-0.01396)∇²f(∇f,∇f)

=== Standard on rotated data ===
f_t = (+0.01920)1 + (-0.12819)f_y + (+0.63322)f_x + (+0.21682)f_yy + (+0.74129)f_xy + (-0.20919)f_xx + (+0.51331)f_yyy + (-0.22060)f_xyy + (-0.96717)f_xxy + (-0.13922)f_xxx + (+0.16770)f_yyyy + (+0.50089)f_xyyy + (-1.20670)f_xxyy + (-0.13648)f_xxxy + (-0.25115)f_xxxx + (+0.00005)f + (-0.00138)f·f_y + (+0.00259)f·f_x + (+0.00092)f·f_yy + (+0.00325)f·f_xy + (-0.00086)f·f_xx + (+0.00212)f·f_yyy + (-0.00082)f·f_xyy + (-0.00476)f·f_xxy + (-0.00072)f·f_xxx + (+0.00074)f·f_yyyy + (+0.00227)f·f_xyyy + (-0.00566)f·f_xxyy + (-0.00076)f·f_xxxy + (-0.00117)f·f_xxxx + (+0.00000)f^2 + (-0.00000)f^2·f_y + (+0.00000)f^2·f_x + (+0.00000)f^2·f_

## 7. Save results

In [10]:
results = {
    'w_eq':  w_eq,  'w_std':  w_std,
    'w_eq_rotated': w_eq_r, 'w_std_rotated': w_std_r,
    'cond_eq':  float(np.linalg.cond(Theta_eq)),
    'cond_std': float(np.linalg.cond(Theta_std)),
    'true_eq':  true_eq,
    'desc_eq':  desc_eq,
    'desc_std': desc_std,
    'coeff_error': coeff_error_eq,
    'false_pos':   false_pos_eq,
    'rot_change_eq':  rot_change_eq,
    'rot_change_std': rot_change_std,
}
np.save('kuramoto_sivashinsky_results.npy', results, allow_pickle=True)
print('Saved kuramoto_sivashinsky_results.npy')


Saved kuramoto_sivashinsky_results.npy
